In [ ]:

# Indlæsning af data


import pandas as pd

# Indlæs datasættet fra CSV-fil
df = pd.read_csv("Data/2026_lolmatches.csv")

# Se de første rækker for at få et overblik
df.head()

C:\Users\Nicol\AppData\Local\Temp\ipykernel_17936\272777035.py:8: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Data/2026_lolmatches.csv")


,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,...,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,LOLTMNT05_171038,complete,NaN,LIT,2026,Winter,0,2026-01-08 17:08:27,1,16.01,...,192.0,-1103.0,-560.0,-28.0,6.0,1.0,4.0,7.0,4.0,4.0
1,LOLTMNT05_171038,complete,NaN,LIT,2026,Winter,0,2026-01-08 17:08:27,1,16.01,...,194.0,-586.0,-1672.0,-34.0,4.0,2.0,5.0,4.0,2.0,3.0
2,LOLTMNT05_171038,complete,NaN,LIT,2026,Winter,0,2026-01-08 17:08:27,1,16.01,...,221.0,-333.0,-807.0,-8.0,1.0,4.0,4.0,5.0,3.0,5.0
3,LOLTMNT05_171038,complete,NaN,LIT,2026,Winter,0,2026-01-08 17:08:27,1,16.01,...,194.0,2641.0,1926.0,38.0,8.0,3.0,4.0,4.0,6.0,4.0
4,LOLTMNT05_171038,complete,NaN,LIT,2026,Winter,0,2026-01-08 17:08:27,1,16.01,...,23.0,1136.0,100.0,3.0,1.0,12.0,3.0,0.0,11.0,4.0


In [ ]:

# Overblik over datastruktur

# Se antal rækker og kolonner (dimensioner)
df.shape

# Se alle kolonnenavne
df.columns

# Få information om datatyper og manglende værdier
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27516 entries, 0 to 27515
Columns: 165 entries, gameid to opp_deathsat25
dtypes: float64(122), int64(20), str(23)
memory usage: 34.6 MB


In [ ]:
# -------------------------------
# Filtrering af relevante deltagere
# -------------------------------

# Behold kun rækker hvor participantid er 100 eller 200
df = df[df["participantid"].isin([100, 200])]


# -------------------------------
# Udvælgelse af relevante kolonner
# -------------------------------

# Opret nyt datasæt med kun de features vi skal bruge
df_model = df[[
    "gameid",
    "side",
    "firstPick",
    "pick1",
    "pick2",
    "pick3",
    "pick4",
    "pick5",
    "result"
]]


# -------------------------------
# Tjek datasættet
# -------------------------------

# Se de første rækker
df_model.head()


,gameid,side,firstPick,pick1,pick2,pick3,pick4,pick5,result
10,LOLTMNT05_171038,Blue,1.0,Xin Zhao,Gnar,Corki,Ahri,Leona,0
11,LOLTMNT05_171038,Red,0.0,Zaahen,Diana,Sivir,Mel,Braum,1
22,LOLTMNT05_172024,Blue,1.0,Corki,Rell,Ekko,Dr. Mundo,Renekton,1
23,LOLTMNT05_172024,Red,0.0,Miss Fortune,Galio,Diana,Zac,Zaahen,0
34,LOLTMNT05_171043,Blue,1.0,Ziggs,Orianna,Nocturne,Ambessa,Alistar,0


In [ ]:
# -----------------------------------
# Opdeling af data i Blue og Red Side
# -----------------------------------

# Filtrér alle rækker hvor siden er Blue
blue = df_model[df_model["side"] == "Blue"]

# Filtrér alle rækker hvor siden er Red
red = df_model[df_model["side"] == "Red"]


# ---------------------
# Sammenkobling af data
# ---------------------

# Merge Blue og Red på gameid
# Så hver kamp bliver samlet i én række
matches = blue.merge(
    red,
    on="gameid",
    suffixes=("_blue", "_red")  # Gør kolonnenavne tydelige
)


# -------------------------------
# Tjek datastruktur
# -------------------------------

# Se de første rækker af det nye datasæt
matches.head()

,gameid,side_blue,firstPick_blue,pick1_blue,pick2_blue,pick3_blue,pick4_blue,pick5_blue,result_blue,side_red,firstPick_red,pick1_red,pick2_red,pick3_red,pick4_red,pick5_red,result_red
0,LOLTMNT05_171038,Blue,1.0,Xin Zhao,Gnar,Corki,Ahri,Leona,0,Red,0.0,Zaahen,Diana,Sivir,Mel,Braum,1
1,LOLTMNT05_172024,Blue,1.0,Corki,Rell,Ekko,Dr. Mundo,Renekton,1,Red,0.0,Miss Fortune,Galio,Diana,Zac,Zaahen,0
2,LOLTMNT05_171043,Blue,1.0,Ziggs,Orianna,Nocturne,Ambessa,Alistar,0,Red,0.0,Wukong,Sivir,Galio,K'Sante,Neeko,1
3,LOLTMNT05_171051,Blue,1.0,Jayce,Cho'Gath,Corki,Akali,Leona,1,Red,0.0,K'Sante,Yunara,Xin Zhao,Orianna,Poppy,0
4,LOLTMNT05_171066,Blue,1.0,Jarvan IV,Ambessa,Ryze,Lucian,Bard,1,Red,0.0,Rumble,Neeko,Wukong,Corki,Rakan,0


In [ ]:
# -------------------------------
# Klargøring af modeldata
# -------------------------------

# Brug resultatet fra Blue side som target
# 1 = Blue vinder, 0 = Blue taber
matches["result"] = matches["result_blue"]


# -------------------------------
# Feature engineering
# -------------------------------

# Tilføj side-information
# Da datasættet er struktureret fra Blue perspektiv,
# vil Blue altid være "1"
matches["blue_side"] = 1


# -------------------------------
# Udvælg features til modellen
# -------------------------------

model_data = matches[[
    "gameid",  
    "pick1_blue","pick2_blue","pick3_blue","pick4_blue","pick5_blue",
    "pick1_red","pick2_red","pick3_red","pick4_red","pick5_red",
    "firstPick_blue",
    "blue_side",
    "result"
]]


# -------------------------------
# Rensning af data
# -------------------------------

# Fjern rækker med manglende værdier
# (nødvendigt for modeller som Logistic Regression)
model_data = model_data.dropna()


# -------------------------------
# Tjek datasættet
# -------------------------------

# Se de første rækker
model_data.head()

# Tjek dimensioner
print("Shape:", model_data.shape)

# Tjek for manglende værdier
print("Missing values:\n", model_data.isnull().sum())

Shape: (2255, 14)
Missing values:
 gameid            0
pick1_blue        0
pick2_blue        0
pick3_blue        0
pick4_blue        0
pick5_blue        0
pick1_red         0
pick2_red         0
pick3_red         0
pick4_red         0
pick5_red         0
firstPick_blue    0
blue_side         0
result            0
dtype: int64


In [ ]:
# -------------------------------
# Rensning af data
# -------------------------------

# Fjern rækker med manglende værdier
df = df_model.dropna()

# Fjern dubletter (hvis der er nogen)
df_model = df_model.drop_duplicates()

# Tjek datasættet efter rensning
df_model.info()

<class 'pandas.DataFrame'>
Index: 4586 entries, 10 to 27515
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   gameid     4586 non-null   str    
 1   side       4586 non-null   str    
 2   firstPick  4526 non-null   float64
 3   pick1      4510 non-null   str    
 4   pick2      4510 non-null   str    
 5   pick3      4510 non-null   str    
 6   pick4      4510 non-null   str    
 7   pick5      4510 non-null   str    
 8   result     4586 non-null   int64  
dtypes: float64(1), int64(1), str(7)
memory usage: 358.3 KB


In [ ]:
# -----------------------------
# Formatering af data
# -----------------------------

# Adskil features (X) og target (y)
# Inputvariabler (draft + side)
X = model_data.drop("result", axis=1)

# Target (kampens udfald)
y = model_data["result"]


# -----------------------------
# Encoding af kategoriske data
# -----------------------------

# Konverter alle kategoriske variabler (champions osv.) til numeriske værdier
# One-hot encoding: hver kategori bliver til en kolonne med 0/1
X = pd.get_dummies(X)


# -----------------------------
# Tjek datastruktur
# -----------------------------

# Se dimensioner (antal rækker og features)
print("X shape:", X.shape)

# Tjek at der IKKE er strings tilbage
print("String columns:", X.select_dtypes(include="object").columns)


X shape: (2255, 3488)
String columns: Index([], dtype='str')


In [ ]:
from sklearn.model_selection import train_test_split

# Split data i træning og test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,   # 20% test
    random_state=42  # gør det reproducerbart
)

In [ ]:
# -------------------------------
# Logistic Regression model
# -------------------------------

from sklearn.linear_model import LogisticRegression

# Initialiser model
# max_iter=1000 sikrer, at modellen får nok iterationer til at konvergere
# (dvs. finde stabile vægte). Standardværdien er ofte for lav ved mange features.
model_lr = LogisticRegression(max_iter=1000)

# Træn modellen på træningsdata
# Her lærer modellen sammenhængen mellem draft (X_train) og kampens udfald (y_train)
model_lr.fit(X_train, y_train)

# Forudsig sandsynligheder
# predict_proba returnerer sandsynligheden for begge klasser [P(0), P(1)]
# [:, 1] vælger sandsynligheden for den positive klasse (win)
y_pred_proba = model_lr.predict_proba(X_test)[:, 1]

# Forudsig klasser (0 eller 1)
# Bruges typisk til accuracy, men ikke til log loss
y_pred = model_lr.predict(X_test)

In [ ]:
# -------------------------------
# Evaluering af model
# -------------------------------

from sklearn.metrics import log_loss, accuracy_score, roc_auc_score

# Log loss (vigtigste metric i dette projekt)
# Straffer usikre og forkerte sandsynligheder hårdt
logloss = log_loss(y_test, y_pred_proba)

# Accuracy (andel korrekte forudsigelser)
# Mindre vigtig ved sandsynlighedsmodeller, men god som supplement
accuracy = accuracy_score(y_test, y_pred)

# ROC AUC (måler hvor godt modellen skelner mellem klasser)
# Tager højde for hele sandsynlighedsfordelingen
roc = roc_auc_score(y_test, y_pred_proba)

# Udskriv resultater
print("Log Loss:", logloss)
print("Accuracy:", accuracy)
print("ROC AUC:", roc)

Log Loss: 0.8773050060536705
Accuracy: 0.5099778270509978
ROC AUC: 0.4940919295758006


In [ ]:
# -----------------------------------------------------------
# Gem encoded draft features til brug i vores samlet model
# ---------------------------------------------------------

# Kopi af features
encoded_draft = X.copy()

# Tilføj target
encoded_draft["result"] = y

# Tilføj gameid
encoded_draft["gameid"] = model_data["gameid"].values

# Gem som CSV fil
encoded_draft.to_csv("Data/draft_encoded.csv", index=False)